# Session-Based Recommendation Algorithms

Companion notebook for the [Session-Based Recommendation Algorithms wiki page](https://ml-viz-ruby.vercel.app/wiki/session-based-recommendations).

We implement GRU4Rec-style update rules, SASRec-style causal attention, and compare BPR vs cross-entropy losses on synthetic session data.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1 — BPR-max loss for session recommendation

In [ ]:
def bpr_max_loss(pos_score, neg_scores):
    """BPR-max: maximize margin between positive item and hardest negative."""
    softmax_neg = np.exp(neg_scores - neg_scores.max())
    softmax_neg /= softmax_neg.sum()
    return -np.log(np.sum(softmax_neg * np.exp(pos_score - neg_scores)) + 1e-9)

# 1 positive item, 5 negative items
pos_score = 0.8
neg_scores = np.array([0.2, 0.5, 0.7, 0.3, 0.4])
loss = bpr_max_loss(pos_score, neg_scores)
print(f"BPR-max loss (pos=0.8, hard neg=0.7): {loss:.4f}")

# When positive is clearly better, loss should be low
pos_easy = 2.0
loss_easy = bpr_max_loss(pos_easy, neg_scores)
print(f"BPR-max loss (pos=2.0, easy): {loss_easy:.4f}")

## 2 — Causal self-attention (SASRec layer)

In [ ]:
def sasrec_layer(E, W_q, W_k, W_v):
    """Single SASRec attention layer. E: (T, d), W_*: (d, d)"""
    T, d = E.shape
    Q, K, V = E @ W_q, E @ W_k, E @ W_v
    scores = Q @ K.T / d**0.5
    mask = np.triu(np.full((T,T), -1e9), k=1)
    attn = np.exp(scores + mask)
    attn /= attn.sum(1, keepdims=True)
    return attn @ V

T, d = 6, 8
E = rng.normal(size=(T, d))
W_q = rng.normal(size=(d, d)) * 0.1
W_k = rng.normal(size=(d, d)) * 0.1
W_v = rng.normal(size=(d, d)) * 0.1

H = sasrec_layer(E, W_q, W_k, W_v)
print(f"Input shape: {E.shape}, Output shape: {H.shape}")
print("Last position embedding (used for next-item prediction):", H[-1].round(3))

## ✏️ Your turn — negative sampling strategy comparison

In [ ]:
def train_loss_with_negatives(pos_scores, neg_strategy, all_scores):
    """
    Compare BPR-max loss under different negative sampling strategies.
    pos_scores: (B,) scores for positive items
    neg_strategy: 'random', 'in_batch', 'hard'
    all_scores: (B, N) scores for all items
    """
    # TODO(you): implement the three strategies and compute mean BPR-max loss
    # random: sample uniformly from all_scores (columns != positive)
    # in_batch: use the other B positive items' scores as negatives
    # hard: use the top-5 scoring items (excluding the positive) as negatives
    return ...

# Test your implementation
B, N = 8, 50
pos_scores = rng.uniform(0.5, 1.5, B)
all_scores = rng.uniform(-1, 2, (B, N))
# The positive item is at column 0 for each row
all_scores[np.arange(B), 0] = pos_scores  # fix the positive positions

for strategy in ['random', 'in_batch', 'hard']:
    loss = train_loss_with_negatives(pos_scores, strategy, all_scores)
    print(f"{strategy:10s}: loss = {loss}")

<details><summary>Hint</summary>

- **random**: for each row, sample k columns ≠ 0, use those scores as negatives in BPR-max.
- **in_batch**: for row i, use `pos_scores[j]` for all j ≠ i as negatives.
- **hard**: for each row, find the top-5 highest-scoring items (excluding col 0) as negatives.

Hard negatives should give the highest loss (hardest to learn from), random the lowest.
</details>